# Rules & systems summary

Point this at a scenario config and it tells you what the world *is*: every
transform, what each resource costs to simply keep existing, where the feedback
loops are, and — after a replay — which rules actually fire versus which are
dead letters.

The centrepiece is an interactive flow DAG with two views over the same layout:

| view | nodes | reads as |
|---|---|---|
| **Transforms** | resources + transforms | exact Petri net — nothing is lost |
| **Resource flow** | resources only | flow diagram; the transforms responsible live in the edge tooltip |

**Hover any edge or node.** Everything else dims and a panel names the
transform(s) responsible, the stoichiometry, the net effect, and how much
actually flowed during the replay.

`analysis.py` derives the facts (pure, no rendering); `dagviz.py` emits
self-contained SVG. Neither touches `sim/`.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))   # repo root, so `import sim` works

from IPython.display import Markdown, display

from sim import load_scenario, scenario_from_dict, initial_world
import analysis
import dagviz
import viz

CONFIG = Path.cwd().parent / "scenarios_data" / "simple-world.json"

## 1. Load the config

Raw JSON alongside the compiled `Scenario` — the compiler drops `description`,
and it is worth showing.

In [ ]:
raw = json.loads(CONFIG.read_text())
scenario = load_scenario(CONFIG)

print(f"{scenario.name}: {scenario.R} resources, {scenario.T} transforms, "
      f"{scenario.L} locations")
print(f"evaluation order: {' -> '.join(raw['evaluation_order'])}")

## 2. Rules & systems

Read the **cheapest hold** column first — under universal decay it is the price of
merely continuing to exist, and it is where scenarios quietly go wrong. Warnings at
the bottom are derived, not hand-written: orphaned resources and feeders evaluated
after their own consumers.

In [ ]:
display(Markdown(analysis.rules_summary(scenario, raw.get("description", ""))))

## 3. Replay

Run the engine forward and record what fired. This is what separates *rules that
exist* from *rules that matter* — and it drives the edge widths in the DAG below.

In [ ]:
runtime = analysis.run_and_observe(scenario, ticks=8, seed=0)
display(Markdown(analysis.observed_summary(scenario, runtime)))

## 4. The flow DAG

Edge width is the volume observed over the replay; dashed edges never fired at
all. Toggle **Resource flow** for the collapsed view, where a self-loop on a
resource is its storage path.

In [ ]:
dagviz.show(scenario, runtime=runtime)

### Structure only

The same graph with no replay attached — pure config, every rule weighted
equally. Useful when designing a scenario before it can survive a single turn.

In [ ]:
dagviz.show(scenario, title="simple-world (static)")

## 5. Location topology

The other DAG: which location may draw from which pool. Node labels are the
initial stock.

In [ ]:
world = initial_world(scenario, seed=0)
viz.draw_world(world, scenario, title="locations at tick 0");

## 6. Compare a variant

Everything above is a function of the config, so a variant is a dict edit away.
Here: evaluate `greenhouse` **before** `habitat`, so the feeder runs its own
production before its consumer drains the shared pool (ISSUES #2). Watch the
warning disappear and the fire counts move.

In [ ]:
variant_raw = {**raw, "evaluation_order": ["greenhouse", "habitat"]}
variant = scenario_from_dict(variant_raw)
variant_runtime = analysis.run_and_observe(variant, ticks=8, seed=0)

display(Markdown(analysis.rules_summary(variant, "feeder evaluated first")))
display(Markdown(analysis.observed_summary(variant, variant_runtime)))

In [ ]:
dagviz.show(variant, runtime=variant_runtime, title="greenhouse-first")

### Side by side

Totals per tick for both evaluation orders.

In [ ]:
import numpy as np

base = runtime.stock_history.sum(axis=1)
alt = variant_runtime.stock_history.sum(axis=1)
header = "tick  " + "".join(f"{r:>9}" for r in scenario.resources)
print("habitat-first" + " " * 20 + "greenhouse-first")
print(header + "   |   " + header)
for t in range(base.shape[0]):
    left = f"{t:<6}" + "".join(f"{int(v):>9}" for v in base[t])
    right = f"{t:<6}" + "".join(f"{int(v):>9}" for v in alt[t])
    print(left + "   |   " + right)